# TSTR Evaluation


In [5]:
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'legend.fontsize': 9,
                     'axes.titlesize': 10})

In [16]:
RESULT_DIR = 'results/tstr'
CLASSES = ['bradypnea', 'eupnea', 'tachypnea']
MODEL_ORDER = ['trtr', 'cvae', 'vae', 'gan']
REGION_ORDER = ['mouth', 'nose']
COLORS = {'trtr': 'tab:blue', 'cvae': 'tab:green', 'vae': 'tab:orange', 'gan': 'tab:red'}
REGION_STYLE = {'mouth': '-', 'nose': '--'}
models = ['cvae', 'vae', 'gan']

# load json files
records = []
for path in glob.glob(f'{RESULT_DIR}/*.json'):
    with open(path) as f:
        records.append(json.load(f))

# load summary
df = pd.read_csv(f'{RESULT_DIR}/summary.csv')
df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
df = df.sort_values(['region', 'model']).reset_index(drop=True)

print(df.to_string(index=False))

model region  accuracy  f1_weighted  roc_auc_ovr  log_loss  f1_bradypnea  f1_eupnea  f1_tachypnea  feature_overlap  n_synthetic
 trtr  mouth  0.928571     0.925897     1.000000  0.225687      0.875000   0.947368      0.952381              NaN          215
 cvae  mouth  0.857143     0.860317     0.948051  0.442466      0.888889   0.800000      0.888889              0.4          215
  vae  mouth  0.142857     0.132937     0.250390  1.215444      0.166667   0.000000      0.222222              0.6          215
 trtr   nose  0.815789     0.815366     0.948761  0.434093      0.846154   0.814815      0.782609              NaN          298
 cvae   nose  0.500000     0.511000     0.694316  1.198421      0.592593   0.571429      0.357143              0.5          298
  vae   nose  0.210526     0.190850     0.340043  1.208041      0.187500   0.370370      0.000000              0.6          298


In [ ]:
metrics_main = ['accuracy', 'f1_weighted', 'roc_auc_ovr', 'log_loss']
metric_labels = ['Accuracy', 'F1 (weighted)', 'ROC-AUC (OvR)', 'Log Loss']

rows = []
for region in REGION_ORDER:
    trtr_row = df[(df.model == 'trtr') & (df.region == region)].iloc[0]
    for model in models:
        r = df[(df.model == model) & (df.region == region)].iloc[0]
        entry = {'model': model, 'region': region}
        for m in metrics_main:
            t = trtr_row[m]
            v = r[m]
            if m == 'log_loss':
                entry[m] = t / v if v > 0 else np.nan
            else:
                entry[m] = v / t if t > 0 else np.nan
        rows.append(entry)

util_df = pd.DataFrame(rows)
fig, axes = plt.subplots(1, len(metrics_main), figsize=(14, 4), sharey=True)
x = np.arange(2)
width = 0.15
for ax, m, label in zip(axes, metrics_main, metric_labels):
    for i, model in enumerate(models):
        vals = [util_df[(util_df.model == model) & (util_df.region == r)][m].values[0] for r in REGION_ORDER]
        bars = ax.bar(x + i * width, vals, width, label=model.upper(), color=COLORS[model], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.axhline(1.0, linewidth=1, label='TRTR baseline')
    ax.set_title(label)
    ax.set_xticks(x + width)
    ax.set_xticklabels(REGION_ORDER)
    ax.set_ylim(0, 1.25)
    ax.set_ylabel('Utility score' if label == metric_labels[0] else '')
    ax.legend(loc='upper right') if m == metrics_main[3] else None
    ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
axes = axes.flatten()
metrics_abs = ['accuracy', 'f1_weighted', 'roc_auc_ovr', 'log_loss']
metric_labels_abs = ['Accuracy', 'F1 (weighted)', 'ROC-AUC (OvR)', 'Log Loss ↓']
invert = [False, False, False, True]
x = np.arange(len(MODEL_ORDER))
width = 0.35
hatch = {'mouth': '', 'nose': '//'}
for ax, m, label, inv in zip(axes, metrics_abs, metric_labels_abs, invert):
    for j, region in enumerate(REGION_ORDER):
        vals = [df[(df.model == mo) & (df.region == region)][m].values[0] for mo in MODEL_ORDER]
        bars = ax.bar(x + j * width, vals, width, label=region, color=[COLORS[mo] for mo in MODEL_ORDER], hatch=hatch[region], alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005 * (ax.get_ylim()[1] if ax.get_ylim()[1] else 1), f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, rotation=0)
    ax.set_title(label)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([m.upper() for m in MODEL_ORDER])
    legend_patches = [mpatches.Patch(facecolor='silver', edgecolor='black', hatch=hatch[r], label=r) for r in REGION_ORDER]
    ax.legend(handles=legend_patches, loc='upper left') if m == metrics_abs[3] else None

model_patches = [mpatches.Patch(facecolor=COLORS[m], label=m.upper()) for m in MODEL_ORDER]
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, region in zip(axes, REGION_ORDER):
    matrix = []
    row_labels = []
    for model in MODEL_ORDER:
        r = df[(df.model == model) & (df.region == region)].iloc[0]
        matrix.append([r['f1_bradypnea'], r['f1_eupnea'], r['f1_tachypnea']])
        row_labels.append(model.upper())

    mat = np.array(matrix)
    im = ax.imshow(mat, vmin=0, vmax=1, cmap='viridis', aspect='equal')

    ax.set_xticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES)
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(row_labels)
    ax.set_title(f'Region: {region}')

    for i in range(len(MODEL_ORDER)):
        for j in range(len(CLASSES)):
            v = mat[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='white' if v < 0.5 else 'black')

plt.colorbar(im, ax=axes[-1], label='F1 score')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
width = 0.3
for i, model in enumerate(models):
    vals = [df[(df.model == model) & (df.region == r)]['feature_overlap'].values[0] for r in REGION_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=model.upper(), color=COLORS[model], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,f'{v:.1%}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(REGION_ORDER)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Overlap')
ax.set_title('Top-10 SHAP Feature Overlap: Synthetic vs Real')
ax.legend()
plt.tight_layout()
plt.show()